# Tutorial 12 — Vision-language models, document understanding, and grounding

[Lecture notes](../lectures/12_vlm.md) · [Exercise sheet](../exercises/12_vlm.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Generate chart pixels, extract bar heights, and audit blank/counterfactual image interventions.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Render controlled chart pixels

The specialized parser knows the bar locations and pixel scale. This is an interpretable baseline for a later VLM comparison, not a learned VLM. Red, green, blue denote A, B, C.

Before running: predict the output or the invariant being checked.


In [2]:
def chart(values):
    im=np.full((64,96,3),255,dtype=np.uint8)
    for i,h in enumerate(values):
        color=np.zeros(3,dtype=np.uint8); color[i]=200
        im[56-int(h)*4:56,8+28*i:24+28*i]=color
    return im

def read_heights(im):
    heights=[]
    for i in range(3):
        column=im[:56,12+28*i]
        colored=(column.min(axis=1)<100)&(column.max(axis=1)>100)
        heights.append(int(colored.sum())/4)
    return np.array(heights)
values=np.array([3,7,5]); original=chart(values)
print("image shape",original.shape,"read heights",read_heights(original))
assert np.array_equal(read_heights(original),values)


image shape (64, 96, 3) read heights [3. 7. 5.]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Intervene on evidence

Use identical questions with original, blank, and counterfactual images. A blank image has no bar evidence; the parser abstains.

Before running: predict the output or the invariant being checked.


In [3]:
def answer(im):
    h=read_heights(im)
    if h.max()==0: return None
    if (h==h.max()).sum()>1: return "tie"
    return "ABC"[int(h.argmax())]
blank=np.full_like(original,255); swapped=chart([7,3,5])
for name,im in [("original",original),("blank",blank),("counterfactual",swapped)]:
    print(name,"highest bar",answer(im))
assert answer(original)=="B" and answer(swapped)=="A" and answer(blank) is None


original highest bar B
blank highest bar None
counterfactual highest bar A


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Break the rendering assumptions

Move the chart sideways. A rule tuned to fixed positions may fail even when the human-visible task is unchanged. Generalization must be measured, not assumed.

Before running: predict the output or the invariant being checked.


In [4]:
shifted=np.roll(original,20,axis=1)
print("shifted heights",read_heights(shifted),"answer",answer(shifted))
print("raw patch counts at patch size 14",(336//14)**2,(672//14)*(336//14))
assert (448//14)**2==1024
# Inspect pixel evidence for the highest bar: colored vertical runs.
print("green-bar center rows",np.where(original[:56,40,1]==200)[0])


shifted heights [0. 3. 7.] answer C
raw patch counts at patch size 14 576 1152
green-bar center rows [28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51
 52 53 54 55]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why is a blank-image baseline useful?
2. Can a correct numerical answer be visually ungrounded?
3. Why might a specialized chart parser outperform a small VLM on this lab?


## Extension and submission

Run `extensions/hf_vision.py --mode vlm` for learned visual question answering; inspect the image, prompt, processor configuration, and generated answer together.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
